In [ ]:
import kagglehub
import os
import pandas as pd

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(os.path.join(path, "Q3_data.csv"))
df.head()

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 1: Write your code here:
import numpy as np

df_clean = df.copy()

num_cols = df_clean.select_dtypes(include=[np.number]).columns
obj_cols = df_clean.select_dtypes(include=['object']).columns

df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
df_clean[obj_cols] = df_clean[obj_cols].fillna("Unknown")


In [ ]:
# Task 2: Write your code here:
df_clean = df_clean.drop_duplicates().copy()


In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

obj_cols = df_clean.select_dtypes(include=['object']).columns
for col in obj_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Identify numeric columns (exclude target later)
scaler = StandardScaler()
df_scaled = df_clean.copy()

# We'll scale all numeric features later after defining X/y to avoid scaling target by mistake.


In [ ]:
# Task 5: Write your code here:
# Detect target column (common names, else last column)
target_candidates = ["target", "Target", "default", "Default", "Credit_Default", "credit_default", "label", "Label", "y"]
target_col = next((c for c in target_candidates if c in df_scaled.columns), df_scaled.columns[-1])

class_counts = df_scaled[target_col].value_counts(normalize=True)
print("Target column:", target_col)
print(class_counts)

is_imbalanced = class_counts.max() > 0.6
print("Imbalanced?" , is_imbalanced)


In [ ]:
!pip -q install catboost


In [ ]:
# Task 1: Write your code here:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
import numpy as np

X = df_scaled.drop(columns=[target_col])
y = df_scaled[target_col]

# Scale features only
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)



In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

use_f1 = is_imbalanced  # if imbalanced -> F1 else Accuracy

for train_idx, val_idx in skf.split(X_scaled, y):
    X_train_fold, X_val_fold = X_scaled[train_idx], X_scaled[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=False
    )

    model.fit(X_train_fold, y_train_fold)
    preds = model.predict(X_val_fold)

    if use_f1:
        scores.append(f1_score(y_val_fold, preds))
    else:
        scores.append(accuracy_score(y_val_fold, preds))

metric_name = "F1" if use_f1 else "Accuracy"
print(f"Average {metric_name} across folds:", np.mean(scores))


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import numpy as np

# Get feature importances
importances = full_model.get_feature_importance()
feat_names = X.columns.to_numpy()

# Sort features
sorted_idx = np.argsort(importances)[::-1]

# Take Top 10 only
top_n = 10
top_idx = sorted_idx[:top_n]

plt.figure(figsize=(10, 5))
plt.barh(feat_names[top_idx][::-1], importances[top_idx][::-1])
plt.xlabel("Importance")
plt.title("Top 10 Feature Importance (CatBoost)")
plt.tight_layout()
plt.show()



In [ ]:
# Task 2: Write your code here:
golden_feature = feat_names[sorted_idx[0]]
print("Golden Feature:", golden_feature)


In [ ]:
# Task Bonus: Write your code here:
X_golden = df_scaled[[golden_feature]].values
X_golden_scaled = StandardScaler().fit_transform(X_golden)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_golden = []

for train_idx, val_idx in skf.split(X_golden_scaled, y):
    X_train_fold, X_val_fold = X_golden_scaled[train_idx], X_golden_scaled[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=False
    )

    model.fit(X_train_fold, y_train_fold)
    preds = model.predict(X_val_fold)

    if use_f1:
        scores_golden.append(f1_score(y_val_fold, preds))
    else:
        scores_golden.append(accuracy_score(y_val_fold, preds))

metric_name = "F1" if use_f1 else "Accuracy"
print(f"Golden-only Average {metric_name}:", np.mean(scores_golden))
